
# Distillation through augmented noise — AlexNet → AlexNet-Half

Data-free knowledge distillation on CIFAR-100 (same pipeline as
`distillation_through_augmented_noise_CIFAR100_optimizedA100(2).ipynb`), with the
**ResNet-34 → ResNet-18** pair replaced by **AlexNet → AlexNet-Half**.

Everything else is unchanged: synthetic noise images (smooth gradient / Perlin /
uniform / Gabor / checkerboard) are pushed through a heavy augmentation stack, the
frozen teacher labels the augmented view on the fly, and the student is trained with
the temperature-scaled KL objective under the same batch-size ramp and
phase-aligned cosine LR schedule.

## Architectures

Taken from the ZSKD supplementary,
[Table 2](https://proceedings.mlr.press/v97/nayak19a/nayak19a-supp.pdf)
(Nayak et al., *Zero-Shot Knowledge Distillation in Deep Networks*, ICML 2019),
cross-checked against the authors' released code
([vcl-iisc/ZSKD](https://github.com/vcl-iisc/ZSKD), `model_alex_full.py` /
`model_alex_half.py`).

> *AlexNet-Half is derived from AlexNet by taking half of the convolutional filters
> and half of the neurons in the fully connected layers, except in the
> classification layer.*

| Layer | Kernel / stride | Output (32×32 in) | AlexNet (teacher) | AlexNet-Half (student) |
|---|---|---|---|---|
| conv1 + ReLU + LRN | 5×5, s1, `SAME` | 32×32 | 48 | 24 |
| maxpool1 + BN | 3×3, s2, `VALID` | 15×15 | — | — |
| conv2 + ReLU + LRN | 5×5, s1, `SAME` | 15×15 | 128 | 64 |
| maxpool2 + BN | 3×3, s2, `VALID` | 7×7 | — | — |
| conv3 + ReLU + BN | 3×3, s1, `SAME` | 7×7 | 192 | 96 |
| conv4 + ReLU + BN | 3×3, s1, `SAME` | 7×7 | 192 | 96 |
| conv5 + ReLU | 3×3, s1, `SAME` | 7×7 | 128 | 64 |
| maxpool5 + BN | 3×3, s2, `VALID` | 3×3 | — | — |
| fc1 + ReLU + dropout(0.5) + BN | — | — | 512 | 256 |
| fc2 + ReLU + dropout(0.5) + BN | — | — | 256 | 128 |
| fc3 (classifier) | — | — | `num_classes` | `num_classes` |

Two notes on porting the reference TensorFlow code to PyTorch:

* **LRN.** TF's `local_response_normalization(depth_radius=2, alpha=1e-4, beta=0.75,
  bias=1.0)` sums over a 5-channel window and does *not* divide `alpha` by the window
  size, while `torch.nn.LocalResponseNorm` does. The port passes `alpha * 5` so the two
  are numerically identical.
* **Init.** The reference initialises every weight from `N(0, 0.01)`. That is kept
  available as `paper_init=True`, but the default here is Kaiming, which trains much more
  reliably without TF-era LR tuning.

## What to expect

AlexNet is a far smaller teacher than ResNet-34 — on CIFAR-100 expect roughly **40–45 %**
teacher accuracy rather than the 76.4 % of the ResNet-34 run, so the distilled student
ceiling is correspondingly lower. The ZSKD architectures were designed for **CIFAR-10**;
set `DATASET = "cifar10"` in the config cell below to run them on the dataset they were
specified for (teacher ≈ 83 %).

## Parameter counts

Built as specified above, the counts are:

| | CIFAR-10 (`num_classes=10`) | CIFAR-100 (`num_classes=100`) |
|---|---|---|
| AlexNet (teacher) | 1,659,178 | 1,682,308 |
| AlexNet-Half (student) | 417,434 | 429,044 |

The teacher matches the 1.65 × 10^6 the paper reports for CIFAR-10. The student does
not: the paper reports 7.23 × 10^5, but halving every conv filter and every hidden FC
width — which is both what the paper's own text says and what the authors' released
`model_alex_half.py` does — gives 4.17 × 10^5. The architecture above follows the text
and the released code; the reported student figure appears to be an error in the paper.

## 1. Data

In [ ]:
import zipfile, os

extract_dir = '/home/vsu/Downloads/cifar100_data'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile('/home/vsu/Downloads/archive_cifar100.zip', 'r') as z:
    z.extractall(extract_dir)

In [ ]:
!mkdir /home/vsu/Downloads/cifar100_data/cifar-100-python
!mv /home/vsu/Downloads/cifar100_data/train \
   /home/vsu/Downloads/cifar100_data/test \
   /home/vsu/Downloads/cifar100_data/meta \
   /home/vsu/Downloads/cifar100_data/file.txt \
   /home/vsu/Downloads/cifar100_data/cifar-100-python/

In [ ]:
!find /home/vsu/Downloads/cifar100_data -maxdepth 3

## 2. AlexNet / AlexNet-Half

PyTorch port of the ZSKD supplementary Table 2 architectures (see the header cell for
the layer table and the two porting notes).

In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset, ConcatDataset


# ==============================================================================
# 1. ZSKD AlexNet / AlexNet-Half for 32x32 CIFAR images
#    (Nayak et al., ICML 2019 -- supplementary Table 2. Torchvision's AlexNet is
#     an ImageNet 224x224 design and downsamples 32x32 inputs to nothing, so the
#     paper's CIFAR variant is used instead.)
# ==============================================================================
class AlexNetCIFAR(nn.Module):
    """AlexNet for 32x32 CIFAR inputs, as specified in the ZSKD supplementary
    (Nayak et al., ICML 2019, Table 2) and the authors' released implementation.

    Block order is the one used by the reference code:
        conv -> ReLU -> LRN -> [maxpool] -> BN
        fc   -> ReLU -> dropout -> BN

    widths = (c1, c2, c3, c4, c5, f1, f2)
        AlexNet      : (48, 128, 192, 192, 128, 512, 256)
        AlexNet-Half : (24,  64,  96,  96,  64, 256, 128)
    AlexNet-Half halves every conv filter count and every FC width except the
    classification layer, which stays at num_classes.
    """

    def __init__(self, widths, num_classes=100, dropout=0.5, paper_init=False):
        super().__init__()
        c1, c2, c3, c4, c5, f1, f2 = widths
        self.widths = widths

        # TF reference: local_response_normalization(depth_radius=2, alpha=1e-4,
        # beta=0.75, bias=1.0) -> window of 2*2+1 = 5 channels, and TF does *not*
        # divide alpha by the window size while torch does (alpha/n). Passing
        # alpha*n makes the two numerically identical.
        def lrn():
            return nn.LocalResponseNorm(size=5, alpha=1e-4 * 5, beta=0.75, k=1.0)

        # 'SAME' padding at stride 1: k=5 -> pad 2, k=3 -> pad 1.
        # Pooling is 3x3 / stride 2 'VALID': 32 -> 15 -> 7 -> 3.
        self.conv1, self.lrn1 = nn.Conv2d(3, c1, 5, 1, 2), lrn()
        self.pool1, self.bn1 = nn.MaxPool2d(3, 2), nn.BatchNorm2d(c1)

        self.conv2, self.lrn2 = nn.Conv2d(c1, c2, 5, 1, 2), lrn()
        self.pool2, self.bn2 = nn.MaxPool2d(3, 2), nn.BatchNorm2d(c2)

        self.conv3, self.bn3 = nn.Conv2d(c2, c3, 3, 1, 1), nn.BatchNorm2d(c3)
        self.conv4, self.bn4 = nn.Conv2d(c3, c4, 3, 1, 1), nn.BatchNorm2d(c4)

        self.conv5, self.pool5 = nn.Conv2d(c4, c5, 3, 1, 1), nn.MaxPool2d(3, 2)
        self.bn5 = nn.BatchNorm2d(c5)

        self.fc1, self.drop1, self.bn6 = nn.Linear(3 * 3 * c5, f1), nn.Dropout(dropout), nn.BatchNorm1d(f1)
        self.fc2, self.drop2, self.bn7 = nn.Linear(f1, f2), nn.Dropout(dropout), nn.BatchNorm1d(f2)
        self.fc3 = nn.Linear(f2, num_classes)

        self._init_weights(paper_init)

    def _init_weights(self, paper_init):
        if paper_init:
            # Exactly the reference TF init: N(0, 0.01) weights, bias 1.0 on
            # conv2/conv4/conv5 (the original AlexNet convention), 0 elsewhere.
            ones = {self.conv2, self.conv4, self.conv5}
            for m in self.modules():
                if isinstance(m, (nn.Conv2d, nn.Linear)):
                    nn.init.normal_(m.weight, std=0.01)
                    nn.init.constant_(m.bias, 1.0 if m in ones else 0.0)
            return
        # Default: Kaiming, which trains far more reliably in torch than the
        # reference's fixed 0.01 std (that init relies on TF-era LR tuning).
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.bn1(self.pool1(self.lrn1(F.relu(self.conv1(x)))))
        x = self.bn2(self.pool2(self.lrn2(F.relu(self.conv2(x)))))
        x = self.bn3(F.relu(self.conv3(x)))
        x = self.bn4(F.relu(self.conv4(x)))
        x = self.bn5(self.pool5(F.relu(self.conv5(x))))
        x = torch.flatten(x, 1)
        x = self.bn6(self.drop1(F.relu(self.fc1(x))))
        x = self.bn7(self.drop2(F.relu(self.fc2(x))))
        return self.fc3(x)


def AlexNet(num_classes=100, **kw):
    """Teacher: the ZSKD 'AlexNet' row of supplementary Table 2."""
    return AlexNetCIFAR((48, 128, 192, 192, 128, 512, 256), num_classes=num_classes, **kw)


def AlexNetHalf(num_classes=100, **kw):
    """Student: the ZSKD 'AlexNet-Half' row -- half the conv filters and half
    the FC neurons, classification layer untouched."""
    return AlexNetCIFAR((24, 64, 96, 96, 64, 256, 128), num_classes=num_classes, **kw)

In [ ]:
# Parameter counts, for the record.
for name, net in [("AlexNet (teacher)", AlexNet(100)), ("AlexNet-Half (student)", AlexNetHalf(100))]:
    n = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"{name:24s} {n:>9,d} trainable params")
del net

## 3. Runtime setup and data loaders

`DATASET` switches between CIFAR-100 (as in the original notebook) and CIFAR-10 (the
dataset the ZSKD architectures were specified for). Nothing else in the notebook needs
to change.

`prepare_cifar_root` handles the layout torchvision insists on: it resolves the data as
`<root>/<base_folder>/<batch files>` with `base_folder` fixed at `cifar-10-batches-py`
or `cifar-100-python`, so pointing `root` at the extracted folder itself — or at a
folder unpacked under any other name — raises *"Dataset not found or corrupted"*. The
helper finds the directory that actually holds the batch files and symlinks it into
place under the name torchvision wants, leaving the files where they are. It also makes
the `mkdir`/`mv` cell in section 1 unnecessary.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.backends.cudnn.benchmark = True  # fixed 32x32 input size -> free speedup

# ---- Runtime tuning ported from the fast AlexNet ZO-PGA pipeline ----------
# Same idea: pick worker/precision settings from the hardware actually
# available, don't touch anything that affects the training math.
import os as _os
_cpu_count = _os.cpu_count() or 2
NUM_WORKERS = max(0, min(16, _cpu_count - 1))
PREFETCH_FACTOR = 2 if NUM_WORKERS <= 2 else (4 if NUM_WORKERS <= 8 else 6)
BF16_OK = device.type == "cuda" and torch.cuda.is_bf16_supported()
torch.set_num_threads(max(1, min(_cpu_count, 32)))
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"NUM_WORKERS={NUM_WORKERS} PREFETCH_FACTOR={PREFETCH_FACTOR} BF16_OK={BF16_OK}")

In [ ]:
import os
import torchvision
import torchvision.transforms as T

# ---- dataset config ----------------------------------------------------------
DATASET = "cifar100"          # "cifar100" (original notebook) or "cifar10" (ZSKD paper)
DOWNLOAD = False              # True lets torchvision fetch the dataset itself
DATA_ROOT = {
    "cifar100": '/home/vsu/Downloads/cifar100_data',
    "cifar10":  '/home/vu-lab03-pc43/Downloads/data/cifar-10-python',
}[DATASET]

if DATASET == "cifar100":
    DATASET_CLS = torchvision.datasets.CIFAR100
    NUM_CLASSES = 100
    CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
    CIFAR_STD = (0.2675, 0.2565, 0.2761)
elif DATASET == "cifar10":
    DATASET_CLS = torchvision.datasets.CIFAR10
    NUM_CLASSES = 10
    CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
    CIFAR_STD = (0.2470, 0.2435, 0.2616)
else:
    raise ValueError(f"unknown DATASET {DATASET!r}")


# torchvision looks for <root>/<base_folder>/<batch files>; the sentinel is one
# file that must be inside base_folder.
_CIFAR_LAYOUT = {
    "cifar10":  ("cifar-10-batches-py", "data_batch_1"),
    "cifar100": ("cifar-100-python",    "train"),
}


def prepare_cifar_root(path, dataset, download=False):
    """Return a `root` laid out the way torchvision expects.

    torchvision resolves the data as <root>/<base_folder>/<batch files>, where
    base_folder is a fixed name ('cifar-10-batches-py' or 'cifar-100-python').
    Archives from Kaggle and friends rarely unpack to exactly that name, and
    pointing `root` straight at the extracted directory is what produces
    "Dataset not found or corrupted".

    This locates the directory that actually holds the batch files and, when it
    is not already named correctly, symlinks it into place under the expected
    name -- the original layout on disk is left untouched.
    """
    base, sentinel = _CIFAR_LAYOUT[dataset]
    path = os.path.abspath(os.path.expanduser(path))

    # Handed the archive itself rather than a directory -> unpack it in place.
    if os.path.isfile(path) and (path.endswith(".tar.gz") or path.endswith(".zip")):
        import shutil
        print(f"Unpacking {path}")
        shutil.unpack_archive(path, os.path.dirname(path))
        path = os.path.dirname(path)

    # Where do the batch files actually live?
    hit = None
    if os.path.isfile(os.path.join(path, sentinel)):
        hit = path
    elif os.path.isdir(path):
        for dirpath, _, files in os.walk(path):
            if sentinel in files:
                hit = dirpath
                break

    if hit is None:
        if download:
            os.makedirs(path, exist_ok=True)
            print(f"No {dataset} files under {path}; torchvision will download into it.")
            return path
        listing = sorted(os.listdir(path))[:12] if os.path.isdir(path) else "<not a directory>"
        raise FileNotFoundError(
            f"Could not find '{sentinel}' at or under {path}\n"
            f"  contents: {listing}\n"
            f"  Point DATA_ROOT at the folder holding the {dataset} batch files, "
            f"or set DOWNLOAD = True."
        )

    root = os.path.dirname(hit)
    if os.path.basename(hit) != base:
        link = os.path.join(root, base)
        if not os.path.lexists(link):
            try:
                os.symlink(hit, link)
            except OSError as e:
                raise RuntimeError(
                    f"{hit} holds the {dataset} batch files but torchvision needs them "
                    f"under a folder named '{base}'. Could not create {link} ({e}); "
                    f"rename the folder to '{base}' by hand."
                ) from e
            print(f"Linked {link} -> {hit}")
    print(f"{dataset}: root={root} ({base}/{sentinel} present)")
    return root


DATA_ROOT = prepare_cifar_root(DATA_ROOT, DATASET, DOWNLOAD)
print(f"{DATASET}: {NUM_CLASSES} classes, root={DATA_ROOT}")

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor()
])

transform_test = T.Compose([
    T.ToTensor()
])

@torch.no_grad()
def evaluate(model, device, loader):
  model.eval()
  correct, total=0.0, 0.0
  for x, y in loader:
    x,y=x.to(device), y.to(device)
    pred=torch.argmax(model(x), dim=1)
    correct+= (pred==y).sum().item()
    total+=y.size(0)
  return 100.0 * correct / total


test_set = DATASET_CLS(root=DATA_ROOT, train=False, download=DOWNLOAD, transform=transform_test)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
train_set = DATASET_CLS(root=DATA_ROOT, train=True, download=DOWNLOAD, transform=transform_train)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

In [ ]:
############ Normalization ##############
def normalize(x, mean, std):
    mean = torch.tensor(mean, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    std = torch.tensor(std, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    return (x - mean) / std

def unnorm(x,mean=CIFAR_MEAN, std=CIFAR_STD):
    mean = torch.tensor(mean, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    std = torch.tensor(std, device=x.device, dtype=x.dtype).view(1, -1, 1, 1)
    return x*std+mean

class normalization(nn.Module):
  def __init__(self,backbone):
    super().__init__()
    self.backbone=backbone
  def forward(self,x):
    y=normalize(x, CIFAR_MEAN, CIFAR_STD)
    return self.backbone(y)

## 4. Teacher

Unlike the ResNet-34 notebook there is no pre-trained AlexNet checkpoint lying around, so
this cell trains one if the checkpoint is missing and loads it otherwise. The recipe is
the notebook's own (SGD + cosine); ZSKD itself used Adam at 1e-3 for 1000 epochs at batch
512, which is also a reasonable choice here.

In [ ]:
import os

# ==============================================================================
# standard supervised teacher training, saving best model on the fly.
# ==============================================================================
def train_teacher(teacher, train_loader, test_loader, device, epochs=100, lr=0.01,
                   ckpt_path=None):
    teacher.to(device)
    opt = torch.optim.SGD(teacher.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ce = nn.CrossEntropyLoss()

    best_acc = 0.0

    for epoch in range(epochs):
        teacher.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = ce(teacher(x), y)
            loss.backward()
            opt.step()
        sched.step()

        if test_loader is not None:
            acc = evaluate(teacher, device, test_loader)

            if (epoch + 1) % 10 == 0:
                print(f"[teacher] epoch {epoch + 1}/{epochs}  test_acc={acc:.2f}%  best_acc={best_acc:.2f}%")

            if acc > best_acc:
                best_acc = acc
                torch.save(teacher.state_dict(), ckpt_path)
                print(f"[teacher] epoch {epoch + 1}/{epochs}  new best acc={acc:.2f}%  -> saved to {ckpt_path}")

    print(f"[teacher] training done. best_acc={best_acc:.2f}%")
    return teacher, best_acc


teacher_ckpt_path = f'/home/vsu/Downloads/AlexNet_{DATASET}.pth'

if not os.path.exists(teacher_ckpt_path):
    teacher = normalization(AlexNet(NUM_CLASSES)).to(device)
    teacher, best_acc = train_teacher(
        teacher, train_loader, test_loader, device,
        epochs=100, lr=0.01, ckpt_path=teacher_ckpt_path
    )
    print(f"Best teacher checkpoint saved -> {teacher_ckpt_path} (test_acc={best_acc:.2f}%)")
else:
    print(f"Found existing teacher checkpoint -> {teacher_ckpt_path}")

## 5. Noise initializers, KD loss, loaders

In [ ]:
########### Initial Noise #########################
# --- Noise initializer for PGA seeding --------------------------------------
def noise_smooth_gradient(n, device, size=32):
    """Smooth linear gradient at a random angle between two random colours."""
    yy, xx = torch.meshgrid(torch.linspace(0, 1, size), torch.linspace(0, 1, size), indexing="ij")
    imgs = torch.zeros(n, 3, size, size)
    for i in range(n):
        angle = random.uniform(0, 2 * math.pi)
        grad = xx * math.cos(angle) + yy * math.sin(angle)
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)  # kept on CPU so DataLoader workers can use it

def _perlin_grid(size, res, device):
    """Single 2D Perlin field, values roughly in [-1, 1]. `size` must be divisible by `res`."""
    assert size % res == 0, "size must be divisible by res"
    d = size // res
    lin = torch.arange(0, res, 1.0 / d)
    gy, gx = torch.meshgrid(lin, lin, indexing="ij")
    grid = torch.stack((gy % 1, gx % 1), dim=-1)             # (size, size, 2)

    angles = 2 * math.pi * torch.rand(res + 1, res + 1)
    grads = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)
    tile = lambda g: g.repeat_interleave(d, 0).repeat_interleave(d, 1)

    g00, g10, g01, g11 = tile(grads[:-1, :-1]), tile(grads[1:, :-1]), tile(grads[:-1, 1:]), tile(grads[1:, 1:])
    n00 = (torch.stack((grid[..., 0],     grid[..., 1]),     -1) * g00).sum(-1)
    n10 = (torch.stack((grid[..., 0] - 1, grid[..., 1]),     -1) * g10).sum(-1)
    n01 = (torch.stack((grid[..., 0],     grid[..., 1] - 1), -1) * g01).sum(-1)
    n11 = (torch.stack((grid[..., 0] - 1, grid[..., 1] - 1), -1) * g11).sum(-1)

    fade = lambda t: 6 * t**5 - 15 * t**4 + 10 * t**3
    t = fade(grid)
    nx0 = torch.lerp(n00, n10, t[..., 0])
    nx1 = torch.lerp(n01, n11, t[..., 0])
    return torch.lerp(nx0, nx1, t[..., 1])


def noise_perlin(n, device, size=32):
    """Perlin noise field per image, colourised the same way as the gradient noises above."""
    imgs = torch.zeros(n, 3, size, size)
    res_options = [2, 4, 8]  # coarse/medium/fine cell grids, all divide 32 evenly
    for i in range(n):
        res = random.choice(res_options)
        grad = _perlin_grid(size, res, device)
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)


def noise_uniform(n, device, size=32):
    """Plain i.i.d. uniform noise -- no spatial smoothness, unlike the gradient/perlin noises."""
    return torch.rand(n, 3, size, size).clamp(0.0, 1.0)

def noise_gabor(n, device, size=32):
    """Gaussian-windowed sinusoidal grating (Gabor patch), random orientation/frequency/phase."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing="ij")
    for i in range(n):
        theta = random.uniform(0, math.pi)
        freq = random.uniform(2.0, 8.0)
        phase = random.uniform(0, 2 * math.pi)
        sigma = random.uniform(0.3, 0.8)
        x_theta = xx * math.cos(theta) + yy * math.sin(theta)
        y_theta = -xx * math.sin(theta) + yy * math.cos(theta)
        gaussian = torch.exp(-(x_theta**2 + y_theta**2) / (2 * sigma**2))
        grating = torch.cos(2 * math.pi * freq * x_theta + phase)
        grad = gaussian * grating
        grad = (grad - grad.min()) / (grad.max() - grad.min() + 1e-8)
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)


def noise_checkerboard(n, device, size=32):
    """Checkerboard with random cell size and random phase offset per image."""
    imgs = torch.zeros(n, 3, size, size)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    for i in range(n):
        block = random.choice([2, 4, 8, 16])
        ox, oy = random.randint(0, block - 1), random.randint(0, block - 1)
        pattern = (((xx + ox) // block) + ((yy + oy) // block)) % 2
        grad = pattern.float()
        color_a, color_b = torch.rand(3, 1, 1), torch.rand(3, 1, 1)
        for c in range(3):
            imgs[i, c] = grad * color_a[c] + (1 - grad) * color_b[c]
    return imgs.clamp(0.0, 1.0)

NOISE_INITIALIZERS = {
    "noise_smooth": noise_smooth_gradient,
    "noise_perlin": noise_perlin,
    "noise_uniform": noise_uniform,
    "noise_gabor": noise_gabor,
    "noise_checkerboard": noise_checkerboard,
}
NOISE_TYPES = list(NOISE_INITIALIZERS.keys())

########## KL Divergence ################
def klpga(x, student, t_logits,T):
  s_logits=student(x)/T
  t_logits=t_logits.detach()
  log_prob= F.log_softmax(s_logits, dim=-1)
  t_prob=F.softmax(t_logits/T, dim=-1)
  kl_dv=F.kl_div(log_prob, t_prob, reduction="batchmean")*T**2
  return kl_dv

############Custom Dataset ####################
class CustomDataset(Dataset):
  def __init__(self,x):
    self.x=x
  def __len__(self):
    return(self.x.size(0))
  def __getitem__(self,idx):
    return self.x[idx]


########################## Dataset to Loader #################
# Cache one DataLoader per batch size so the (intentional) 16 -> 2048 batch-size
# ramp doesn't respawn the worker pool every single epoch. The ramp itself is
# unchanged; only the loader lifecycle is optimized.
_loader_cache = {}

def data_to_loader(data, i):
    if i < 25:
        batch_size = 16
    elif i<50:
        batch_size=64
    elif i<75:
        batch_size=128
    elif i<100:
        batch_size=256
    elif i<125:
        batch_size=512
    elif i<150:
        batch_size=1024
    else:
      batch_size=2048

    if batch_size not in _loader_cache:
        num_workers = NUM_WORKERS  # hardware-aware, set up in the setup cell
        kwargs = dict(
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=True,
            drop_last=True,   # keeps batch shape constant -> cudnn.benchmark stays hot
        )
        if num_workers > 0:
            kwargs["persistent_workers"] = True
            kwargs["prefetch_factor"] = PREFETCH_FACTOR
        _loader_cache[batch_size] = DataLoader(data, **kwargs)
    return _loader_cache[batch_size]

## 6. Augmentation

In [ ]:
# =============================================================================
# Augmentation
#
#   Two INDEPENDENT switches, applied in this order:
#     1. geometric base  : RandomCrop(32, pad=4, reflect) + RandomHorizontalFlip
#     2. random op stack : k ops sampled without replacement from the 17-op pool
#
#   Both stages happen inside Dataset.__getitem__, i.e. BEFORE the tensor is
#   handed to the teacher, so the teacher is queried on the augmented view.
# =============================================================================
base_geo_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

_perspective_tf = T.RandomPerspective(distortion_scale=0.35, p=1.0)
_zoom_crop_tf = T.RandomResizedCrop(32, scale=(0.65, 1.0), ratio=(0.85, 1.15))
_color_jitter_tf = T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.45, hue=0.12)
_cutout_tf = T.RandomErasing(p=1.0, scale=(0.02, 0.25), ratio=(0.3, 3.3), value=0.0)


def op_rotate(img):
    return T.functional.rotate(img, random.uniform(-20, 20))


def op_affine(img):
    t = 2
    return T.functional.affine(
        img,
        angle=random.uniform(-12, 12),
        translate=(random.randint(-t, t), random.randint(-t, t)),
        scale=random.uniform(0.82, 1.18),
        shear=random.uniform(-12, 12),
    )


def op_perspective(img):
    return _perspective_tf(img)


def op_zoom_crop(img):
    return _zoom_crop_tf(img)


def op_color_jitter(img):
    return _color_jitter_tf(img)


def op_grayscale(img):
    return T.functional.rgb_to_grayscale(img, num_output_channels=3)


def op_gaussian_blur(img):
    k = random.choice([3, 5])
    return T.functional.gaussian_blur(img, kernel_size=k, sigma=random.uniform(0.1, 2.2))


def op_sharpness(img):
    return T.functional.adjust_sharpness(img, random.uniform(0.0, 3.5))


def op_autocontrast(img):
    return T.functional.autocontrast(img.clamp(0.0, 1.0))


def op_equalize(img):
    img_u8 = (img.clamp(0.0, 1.0) * 255).to(torch.uint8)
    return T.functional.equalize(img_u8).float() / 255.0


def op_posterize(img):
    img_u8 = (img.clamp(0.0, 1.0) * 255).to(torch.uint8)
    return T.functional.posterize(img_u8, random.choice([3, 4, 5, 6])).float() / 255.0


def op_solarize(img):
    return T.functional.solarize(img.clamp(0.0, 1.0), random.uniform(0.3, 0.9))


def op_invert(img):
    return T.functional.invert(img.clamp(0.0, 1.0))


def op_gaussian_noise(img):
    sigma = random.uniform(0.01, 0.08)
    return (img + torch.randn_like(img) * sigma).clamp(0.0, 1.0)


def op_salt_pepper(img):
    prob = random.uniform(0.01, 0.06)
    mask = torch.rand(1, img.shape[1], img.shape[2], device=img.device)
    salt = (mask < prob / 2).expand_as(img)
    pepper = (mask > 1 - prob / 2).expand_as(img)
    out = img.clone()
    out[salt] = 1.0
    out[pepper] = 0.0
    return out


def op_cutout(img):
    return _cutout_tf(img.unsqueeze(0)).squeeze(0)


def op_random_color_erase(img):
    out = img.clone()
    h, w = img.shape[1], img.shape[2]
    eh, ew = random.randint(4, 12), random.randint(4, 12)
    y0 = random.randint(0, h - eh)
    x0 = random.randint(0, w - ew)
    out[:, y0:y0 + eh, x0:x0 + ew] = torch.rand(3, 1, 1, device=img.device)
    return out


# The 17-op pool.
AUG_OPS = {
    "rotate": op_rotate,
    "affine": op_affine,
    "perspective": op_perspective,
    "zoom_crop": op_zoom_crop,
    "color_jitter": op_color_jitter,
    "grayscale": op_grayscale,
    "gaussian_blur": op_gaussian_blur,
    "sharpness": op_sharpness,
    "autocontrast": op_autocontrast,
    "equalize": op_equalize,
    "posterize": op_posterize,
    "solarize": op_solarize,
    "invert": op_invert,
    "gaussian_noise": op_gaussian_noise,
    "salt_pepper": op_salt_pepper,
    "cutout": op_cutout,
    "random_color_erase": op_random_color_erase,
}
AUG_OP_NAMES = list(AUG_OPS.keys())
N_AUG_OPS = len(AUG_OP_NAMES)
assert N_AUG_OPS == 17, f"expected a 17-op pool, found {N_AUG_OPS}"

AUG_QUERY_NOTE = (
    "Teacher is queried on the AUGMENTED view (geo + ops), so crop/flip is "
    "inside the query path."
)


def resolve_op_pool(enable=None, disable=None):
    """Restrict the 17-op pool. `enable`/`disable` are comma-separated name lists."""
    names = list(AUG_OP_NAMES)
    if enable:
        wanted = [s.strip() for s in enable.split(",") if s.strip()]
        unknown = [w for w in wanted if w not in AUG_OPS]
        if unknown:
            raise ValueError(f"unknown aug ops: {unknown}. Valid: {AUG_OP_NAMES}")
        names = [n for n in names if n in wanted]
    if disable:
        drop = {s.strip() for s in disable.split(",") if s.strip()}
        unknown = [d for d in drop if d not in AUG_OPS]
        if unknown:
            raise ValueError(f"unknown aug ops: {unknown}. Valid: {AUG_OP_NAMES}")
        names = [n for n in names if n not in drop]
    return names


def diverse_augment(img, use_geo=True, n_random_ops=4):
    """
    Independent switches:
      use_geo      -> apply RandomCrop(pad 4, reflect) + RandomHorizontalFlip
      n_random_ops -> how many ops to sample (without replacement) from op_pool
    """
    img = img.clamp(0.0, 1.0)
    if use_geo:
        img = base_geo_transform(img)
        pool = AUG_OP_NAMES
        k = min(int(n_random_ops), len(pool))
        for name in random.sample(pool, k=k):
            img = AUG_OPS[name](img)
            img = img.clamp(0.0, 1.0)
    return img


# =============================================================================
# Datasets
# =============================================================================
class SyntheticDataset(Dataset):
    """Yields the augmented noise image; the teacher labels it on the fly."""

    def __init__(self, imgs, use_geo=True, n_random_ops=4):
        self.imgs = imgs
        self.use_geo=use_geo
        self.n_random_ops=n_random_ops

    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, idx):
        img = diverse_augment(self.imgs[idx], self.use_geo, self.n_random_ops)
        return img

In [ ]:
############ Data from Scratch ##############
def noise_data(p):
    all_imgs = []
    for i in range(p):
        noise_fn = random.choice([noise_smooth_gradient, noise_perlin, noise_uniform, noise_gabor, noise_checkerboard])
        imgs = noise_fn(100, device, size=32)
        all_imgs.append(imgs)
    return torch.cat(all_imgs, dim=0)

## 7. Teacher and student

This is the one substantive swap versus the original notebook: `ResNet34()`/`ResNet18()`
become `AlexNet()`/`AlexNetHalf()`.

In [ ]:
################ Teacher and Student Models ##############################

teacher_backbone = AlexNet(NUM_CLASSES).to(device)
student_backbone = AlexNetHalf(NUM_CLASSES).to(device)
teacher = normalization(teacher_backbone)
student = normalization(student_backbone)

teacher_path = teacher_ckpt_path
teacher.load_state_dict(torch.load(teacher_path, map_location=device))
teacher.eval()
for p in teacher.parameters():
  p.requires_grad_(False)
evaluate(teacher, device, test_loader)

## 8. LR schedule and student distillation

In [ ]:
import math

def batch_aligned_lr(epoch, total_epochs=100):
    """Cosine decay within each phase, resetting at every batch-size change
       (epochs 25, 50, 75, 100, 125, 150) to line up with the batch-size ramp."""
    if epoch < 25:
        start, end, phase_start, phase_len = 0.01, 0.00001, 0, 25
    elif epoch < 50:
          start, end, phase_start, phase_len = 0.001, 0.00001, 25, 25
    elif epoch < 75:
         start, end, phase_start, phase_len = 0.01, 0.00001, 50, 25
    elif epoch < 100:
         start, end, phase_start, phase_len = 0.01, 0.00001, 75, 25
    elif epoch < 125:
         start, end, phase_start, phase_len = 0.01, 0.00001, 100, 25
    elif epoch < 150:
         start, end, phase_start, phase_len = 0.01, 0.00001, 125, 25
    else:
        start, end, phase_start, phase_len = 0.01, 0.00001, 150, 50

    t = epoch - phase_start
    cos_factor = 0.5 * (1 + math.cos(math.pi * t / max(phase_len - 1, 1)))
    lr = end + (start - end) * cos_factor
    return lr / 0.01  # normalized multiplier for LambdaLR

In [ ]:
####################################################################
# Student Training
##################################################################
def _fused_sgd_kwargs(device):
    # Fused CUDA kernel for the optimizer step; same math, same update rule.
    import inspect
    if device.type == "cuda" and "fused" in inspect.signature(torch.optim.SGD.__init__).parameters:
        return {"fused": True}
    return {}

def _prepare_for_fast(model, device):
    # channels_last is a pure memory-layout change (NHWC vs NCHW); conv math
    # is identical but cudnn kernels for it run faster on modern GPUs.
    if device.type == "cuda":
        return model.to(memory_format=torch.channels_last)
    return model

def _fast_input(x, device):
    if device.type == "cuda" and x.dim() == 4:
        return x.contiguous(memory_format=torch.channels_last)
    return x

def _train_autocast(device):
    # bf16 when the GPU supports it (no GradScaler needed -- bf16 has fp32's
    # exponent range so it doesn't underflow like fp16). Falls back to fp16
    # autocast+scaler on older GPUs (e.g. T4), or plain fp32 on CPU.
    if device.type == "cuda" and BF16_OK:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16), False
    if device.type == "cuda":
        return torch.amp.autocast("cuda"), True
    from contextlib import nullcontext
    return nullcontext(), False


def train_student(teacher, student, dataset, test_loader, device, T, student_epochs=100, lr=0.001):
  student = _prepare_for_fast(student, device)
  teacher = _prepare_for_fast(teacher, device)
  teacher.eval()   # AlexNet has dropout as well as BN -- keep the labeller deterministic
  opt = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4,
                         **_fused_sgd_kwargs(device))
  sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=batch_aligned_lr)

  autocast_ctx, needs_scaler = _train_autocast(device)
  scaler = torch.amp.GradScaler("cuda", enabled=needs_scaler)
  for epoch in range(student_epochs):
    student.train()
    loader=data_to_loader(dataset,epoch)
    epoch_loss = 0.0
    for x in loader:
      x = _fast_input(x.to(device, non_blocking=True), device)
      with torch.no_grad(), autocast_ctx:  # teacher params already frozen; no_grad
        z = teacher(x)                     # also stops autograd tracking the input side
      opt.zero_grad(set_to_none=True)
      with autocast_ctx:
        loss = klpga(x, student, z, T)
      if needs_scaler:
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
      else:
        loss.backward()
        opt.step()
      epoch_loss += loss.item()
    sched.step()
    accuracy=evaluate(student, device, test_loader)
    print(accuracy)
    if (epoch + 1) % 25 == 0 or epoch == student_epochs - 1:
      print(f"epoch {epoch+1}/{student_epochs}  loss={epoch_loss/len(loader):.4f}")

  return student

## 9. Run

In [ ]:
data=noise_data(1500)
len(data)
print(data.shape)

In [ ]:
dataset= SyntheticDataset(data, use_geo=True, n_random_ops=8)

In [ ]:
train_student(teacher, student, dataset, test_loader, device, T=20, student_epochs=200, lr=0.01)